In [11]:
import os
import numpy as np
import tifffile
import imagecodecs

def process_nailfold_images(input_folder):
    output_folder = os.path.join(input_folder, "processed_tiffs")
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(".dng"):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename.replace(".dng", "_linear.tif"))
            
            try:
                # We use 'key=0' to grab the first image data block and bypass 
                # metadata parsing that causes the Orientation error.
                with tifffile.TiffFile(input_path) as tif:
                    # Access the raw array directly from the first page
                    # This bypasses the faulty Tag 274
                    raw_data = tif.pages[0].asarray()

                # Samsung Expert RAW DNGs are usually 3-channel Linear DNGs
                if len(raw_data.shape) == 3:
                    # Extract Green channel (Index 1) for Hemoglobin analysis
                    green_linear = raw_data[:, :, 1]
                else:
                    green_linear = raw_data

                # Save as 16-bit TIFF
                tifffile.imwrite(
                    output_path, 
                    green_linear.astype(np.uint16), 
                    photometric='minisblack'
                )
                print(f"Processed: {filename}")

            except Exception as e:
                print(f"Error processing {filename}: {e}")



In [ ]:
image_folder = "../../Mobile NFC Captured Data/6-25-26 PCB Data v1.1/6"
process_nailfold_images(image_folder)

In [ ]:
# import tifffile
# import imageio.v3 as iio

# def normalize_for_view(img, p_low=1, p_high=99.5, gamma=1.0):
#     img = img.astype(np.float32)

#     lo, hi = np.percentile(img, [p_low, p_high])
#     img = np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)

#     if gamma != 1.0:
#         img = img ** (1 / gamma)

#     return (img * 255).astype(np.uint8)


# def process_nailfold_images(input_folder):
#     output_folder = os.path.join(input_folder, "processed_tiffs")
#     png_folder = os.path.join(input_folder, "processed_pngs")

#     os.makedirs(output_folder, exist_ok=True)
#     os.makedirs(png_folder, exist_ok=True)

#     print("Input folder:", os.path.abspath(input_folder))
#     print("TIFF output:", os.path.abspath(output_folder))
#     print("PNG output:", os.path.abspath(png_folder))

#     for filename in os.listdir(input_folder):
#         if filename.lower().endswith(".dng"):
#             input_path = os.path.join(input_folder, filename)

#             base = os.path.splitext(filename)[0]
#             output_tiff_path = os.path.join(output_folder, base + "_green_linear.tif")
#             output_png_path = os.path.join(png_folder, base + "_green_view.png")

#             try:
#                 with tifffile.TiffFile(input_path) as tif:
#                     raw_data = tif.pages[0].asarray()

#                 print(filename, raw_data.shape, raw_data.dtype,
#                       raw_data.min(), raw_data.max())

#                 # If already RGB/linear DNG
#                 if raw_data.ndim == 3:
#                     green_linear = raw_data[:, :, 1]

#                 # If raw Bayer/CFA 2D image
#                 else:
#                     # Simple green extraction from Bayer-like pattern
#                     # This keeps only approximate green pixels.
#                     g1 = raw_data[0::2, 1::2]
#                     g2 = raw_data[1::2, 0::2]

#                     h = min(g1.shape[0], g2.shape[0])
#                     w = min(g1.shape[1], g2.shape[1])
#                     green_linear = ((g1[:h, :w].astype(np.float32) +
#                                      g2[:h, :w].astype(np.float32)) / 2)

#                 # Save true linear 16-bit TIFF for analysis
#                 tifffile.imwrite(
#                     output_tiff_path,
#                     green_linear.astype(np.uint16),
#                     photometric="minisblack"
#                 )

#                 # Save normalized 8-bit PNG for viewing
#                 green_view = normalize_for_view(
#                     green_linear,
#                     p_low=1,
#                     p_high=99.5,
#                     gamma=1.0
#                 )
#                 iio.imwrite(output_png_path, green_view)

#                 print(f"Processed: {filename}")

#             except Exception as e:
#                 print(f"Error processing {filename}: {e}")